# O pipeline em produção

## Task 4: colocar o pipeline em produção, rodando fora do notebook, e carregar a Copa de 2026

O notebook é para explorar. Em produção, o job **roda sozinho**, chamado por um agendador, e é **versionado** como
código.

A célula abaixo já vem pronta: o mesmo código do vídeo 5.1 (`ler_partidas` e o histórico).

In [ ]:
# vídeo 5.1: as partidas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

## O script do pipeline

1. **ler** o CSV de partidas; 2. **transformar** com as funções da aula 2; 3. **gravar** Parquet por ano, com overwrite
dinâmico. As pastas chegam como parâmetros (`--raw` e `--out`).

- **Sem `master`**: quem executa decide onde o job roda.
- **`partitionOverwriteMode=dynamic`** na sessão: nenhuma escrita apaga o histórico.
- **`argparse`** lê os parâmetros da linha de comando.

## Executando com spark-submit

`spark-submit` envia um script para o Spark executar. No terminal do Jupyter, em `~/work`:

```bash
spark-submit --master "local[*]" src/wc_pipeline/pipeline.py --raw data/raw --out data/out/producao
```

A próxima célula já vem pronta: ela só apaga a saída de produção anterior.

In [ ]:
import shutil
shutil.rmtree(f"{OUT}/producao", ignore_errors=True)

## A Copa de 2026

Antes de carregar, a auditoria de nulos do vídeo 2.2:

2026 chegou **sem público** (104 nulos), e o pipeline não quebra. As agregações ignoram os nulos (aula 2).

A carga, **duas vezes**:

666 partidas nas duas execuções: **idempotente**.

## Conferindo a saída

Como no vídeo 2.3, `assert` para conferir, agora lendo a saída do pipeline:

## Onde gravar em produção

| Lugar | Exemplo de caminho |
|---|---|
| disco local | `data/out/producao` |
| armazenamento na nuvem | `s3a://bucket/caminho` (AWS), `gs://` (Google), `abfss://` (Azure) |
| catálogo | `saveAsTable("banco.tabela")` (vídeo 3.4) |

**Nunca escreva chaves de acesso no código.**

## Demanda da aula 5 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | política de escrita definida |
| Task 2 | Parquet |
| Task 3 | saída por Copa, um arquivo por pasta, overwrite dinâmico |
| Task 4 | `pipeline.py` com `spark-submit`, 2026 carregada sem duplicar, saída conferida |

## O projeto completo

| Aula | Entrega |
|---|---|
| 1 | ambiente no ar e arquivos lidos com o schema do projeto |
| 2 | limpeza e métricas em `src/wc_pipeline/transformacoes.py` |
| 3 | views, consulta em `sql/`, rankings e uma tabela no catálogo |
| 4 | plano lido, configuração herdada removida e `docs/otimizacao.md` |
| 5 | `src/wc_pipeline/pipeline.py`, Parquet por Copa e a Copa de 2026 |

Próximo passo: a atividade "Subindo o projeto no Git".

## Para consultar depois

- **Opções do `spark-submit`:** `--conf chave=valor` muda uma configuração só naquela execução, e
  `spark-submit --help` lista as outras.
- **Agendadores:** o Airflow, por exemplo, tem um operador que chama o `spark-submit` com os parâmetros de cada dia.
- **Conferências maiores:** em projetos grandes, as conferências com `assert` viram uma suíte de testes automatizados,
  rodada antes de cada mudança no código.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()